# The Price is Right

Today we build a more complex solution for estimating prices of goods.

1. This notebook: create a RAG database with our 400,000 training data
2. Day 2.1 notebook: visualize in 2D
3. Day 2.2 notebook: visualize in 3D
4. Day 2.3 notebook: build and test a RAG pipeline with GPT-4o-mini
5. Day 2.4 notebook: (a) bring back our Random Forest pricer (b) Create a Ensemble pricer that allows contributions from all the pricers

Phew! That's a lot to get through in one day!

## PLEASE NOTE:

We already have a very powerful product estimator with our proprietary, fine-tuned LLM. Most people would be very satisfied with that! The main reason we're adding these extra steps is to deepen your expertise with RAG and with Agentic workflows.


In [1]:
# imports

import os
from tqdm import tqdm
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import chromadb


In [2]:
# environment

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')
DB = "products_vectorstore"

In [3]:
# Log in to HuggingFace

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Back to the pkl files

Much as we enjoyed data curating in week 6, we probably don't want to go through that whole process again!

Let's reuse the pkl files we created then. Either copy the files `train.pkl` and `test.pkl` from the Week 6 folder into this Week 8 folder, or you can also download them from here:

https://drive.google.com/drive/folders/1f_IZGybvs9o0J5sb3xmtTEQB3BXllzrW?usp=drive_link

In [4]:
# With train.pkl in this folder, you can run this:

with open('../week6/train_lite.pkl', 'rb') as file:
    train = pickle.load(file)

In [5]:
print(len(train))
train[0].prompt

100000


'How much does this cost to the nearest dollar?\n\nBOX USA Jumbo Cable Ties, 175#, 60, Natural (Pack of 50)\n60 175# Jumbo Cable Ties - Natural. Nylon Cable Ties permanently secure cords, cables, bags, etc. Heavy-Duty ties for demanding applications! Nylon ties lock tight! - Will not slide or loosen. Nylon Cable Ties permanently secure cords, cables, bags, etc. Heavy-Duty ties for demanding applications! Nylon ties lock tight! - Will not slide or loosen. Country of origin China Material Nylon, Brand BOX USA, Dimensions LxWxH 60 x 0.35 inches, Closure Type Buckle, Pieces 50, model number Available April 21, 2016, Manufacturer BOX USA, Country of Origin China\n\nPrice is $100.00'

# Now create a Chroma Datastore

In Week 5, we created a Chroma datastore with 123 documents representing chunks of objects from our fictional company Insurellm.

Now we will create a Chroma datastore with 400,000 products from our training dataset! It's getting real!

Note that we won't be using LangChain, but the API is very straightforward and consistent with before.

Special note: if Chroma crashes and you're a Windows user, you should try rolling back to an earlier version of the Chroma library with:  
`!pip install chromadb==0.5.0`  
With many thanks to student Kelly Z. for finding this out and pointing to the GitHub issue [here](https://github.com/chroma-core/chroma/issues/2513). 

In [6]:
client = chromadb.PersistentClient(path=DB)

In [7]:
# Check if the collection exists and delete it if it does
collection_name = "products"
existing_collection_names = [collection.name for collection in client.list_collections()]
if collection_name in existing_collection_names:
    client.delete_collection(collection_name)
    print(f"Deleted existing collection: {collection_name}")

collection = client.create_collection(collection_name)

# Introducing the SentenceTransfomer

The all-MiniLM is a very useful model from HuggingFace that maps sentences & paragraphs to a 384 dimensional dense vector space and is ideal for tasks like semantic search.

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

It can run pretty quickly locally.

Last time we used OpenAI embeddings to produce vector embeddings. Benefits compared to OpenAI embeddings:
1. It's free and fast!
3. We can run it locally, so the data never leaves our box - might be useful if you're building a personal RAG


In [8]:
# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
model = SentenceTransformer("all-MiniLM-L6-v2")  # much faster on CPU

In [9]:
# Pass in a list of texts, get back a numpy array of vectors

vector = model.encode(["Well hi there"])[0]

In [10]:
vector

array([-9.46715698e-02,  4.27619778e-02,  5.51620424e-02, -5.11032064e-04,
        1.16202934e-02, -6.80130571e-02,  2.76405979e-02,  6.06974177e-02,
        2.88530551e-02, -1.74128190e-02, -4.94346581e-02,  2.30993386e-02,
       -1.28614390e-02, -4.31403145e-02,  2.17509400e-02,  4.26549129e-02,
        5.10500148e-02, -7.79727399e-02, -1.23247243e-01,  3.67455967e-02,
        4.54113353e-03,  9.47937295e-02, -5.53098097e-02,  1.70641355e-02,
       -2.92872824e-02, -4.47124764e-02,  2.06784401e-02,  6.39319867e-02,
        2.27428414e-02,  4.87790741e-02, -2.33501848e-03,  4.72859181e-02,
       -2.86259130e-02,  2.30624061e-02,  2.45130006e-02,  3.95681635e-02,
       -4.33176160e-02, -1.02316618e-01,  2.79866788e-03,  2.39304788e-02,
        1.61556099e-02, -8.99079815e-03,  2.07256209e-02,  6.40123338e-02,
        6.89179003e-02, -6.98361844e-02,  2.89760996e-03, -8.10989067e-02,
        1.71123352e-02,  2.50653597e-03, -1.06529132e-01, -4.87733521e-02,
       -1.67761408e-02, -

In [11]:
def description(item):
    text = item.prompt.replace("How much does this cost to the nearest dollar?\n\n", "")
    return text.split("\n\nPrice is $")[0]

In [12]:
description(train[0])

'BOX USA Jumbo Cable Ties, 175#, 60, Natural (Pack of 50)\n60 175# Jumbo Cable Ties - Natural. Nylon Cable Ties permanently secure cords, cables, bags, etc. Heavy-Duty ties for demanding applications! Nylon ties lock tight! - Will not slide or loosen. Nylon Cable Ties permanently secure cords, cables, bags, etc. Heavy-Duty ties for demanding applications! Nylon ties lock tight! - Will not slide or loosen. Country of origin China Material Nylon, Brand BOX USA, Dimensions LxWxH 60 x 0.35 inches, Closure Type Buckle, Pieces 50, model number Available April 21, 2016, Manufacturer BOX USA, Country of Origin China'

### Vector DB - chroma.sqlite3 will be saved to **'products_vectorstore'** folder

In [13]:
# for i in tqdm(range(0, len(train), 1000)):
#     documents = [description(item) for item in train[i: i+1000]]
#     vectors = model.encode(documents).astype(float).tolist()
#     metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
#     ids = [f"doc_{j}" for j in range(i, i+1000)]
#     collection.add(
#         ids=ids,
#         documents=documents,
#         embeddings=vectors,
#         metadatas=metadatas
#     )


### Finetune for Surfacr Pro 11

In [14]:
from concurrent.futures import ThreadPoolExecutor, as_completed

BATCH_SIZE = 1000
MAX_WORKERS = 6  # Safe level for 12-core machine

def process_batch(start_idx):
    batch = train[start_idx:start_idx + BATCH_SIZE]
    documents = [description(item) for item in batch]
    vectors = model.encode(documents, batch_size=32, show_progress_bar=False).astype(float).tolist()
    metadatas = [{"category": item.category, "price": item.price} for item in batch]
    ids = [f"doc_{j}" for j in range(start_idx, start_idx + len(batch))]
    return ids, documents, vectors, metadatas

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_batch, i) for i in range(0, len(train), BATCH_SIZE)]

    for future in tqdm(as_completed(futures), total=len(futures)):
        ids, documents, vectors, metadatas = future.result()
        collection.add(
            ids=ids,
            documents=documents,
            embeddings=vectors,
            metadatas=metadatas
        )


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [5:17:01<00:00, 190.22s/it]
